# Metabolomics overview plot

Date created: 12/19/2024

Notebook author: Yang Chen

Data analysis by: Britta De Pessemier

This notebook plots the following:
- Metabolomics overview plots showing number of total metabolites, with and without suspects library, and classified annotations

In [139]:
# Import Python packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3
import plotly.graph_objects as go
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib import colormaps
from upsetplot import from_indicators, plot
from upsetplot import from_memberships, UpSet


In [140]:
# Read in table of all spectra and output number of total metabolites
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')
total_num_metabolites = info_feature_complete.shape[0]
print(f'Total number of consensus MS2 spectra metabolites from GNPS2: ' + str(total_num_metabolites))

Total number of consensus MS2 spectra metabolites from GNPS2: 7683


In [141]:
# Read in GNPS2 metabolites table obtained WITHOUT suspects library
gnps_without_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps_withoutsuspects.tsv', sep='\t')
num_gnps_without_suspects = gnps_without_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITHOUT suspects library: ' + str(num_gnps_without_suspects))

Number of GNPS2 outputed metabolites WITHOUT suspects library: 432


In [142]:
# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')
num_gnps_with_suspects = gnps_with_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITH suspects library: ' + str(num_gnps_with_suspects))

Number of GNPS2 outputed metabolites WITH suspects library: 1027


In [143]:
# Count rows where 'superclass' is not NaN
num_annotated_metabolites = gnps_with_suspects['superclass'].notna().sum()

print(f"Number of superclass annotated metabolites: {num_annotated_metabolites}")


Number of superclass annotated metabolites: 250


### Nested Venn Diagram showing number of metabolites from GNPS

In [144]:
# Define circle sizes
size1 = total_num_metabolites  # Size of Circle 1
size2 = num_gnps_with_suspects   # Size of Circle 2
size3 = num_gnps_without_suspects   # Size of Circle 3
size4 = num_annotated_metabolites   # Size of Circle 4

# Calculate the radii based on the square root of the sizes (scaled for visualization)
scaling_factor = 0.005  # Adjust this value for appropriate circle size in the plot

radius1 = np.sqrt(size1) * scaling_factor  # Radius for Circle 1
radius2 = np.sqrt(size2) * scaling_factor  # Radius for Circle 2
radius3 = np.sqrt(size3) * scaling_factor  # Radius for Circle 3
radius4 = np.sqrt(size4) * scaling_factor  # Radius for Circle 4

# Calculate the percentage for each circle relative to the total size
total_size = size1
percent1 = (size2 / size1) * 100
percent2 = (size3 / size1) * 100
percent3 = (size4 / size1) * 100

# Create a figure
fig, ax = plt.subplots()

# Choose aesthetically pleasing colors (pastel-like colors)
circle1 = plt.Circle((0.5, 0.5), radius1, facecolor='#000000', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size1} total MS2 spectra identified')
circle2 = plt.Circle((0.5, 0.22), radius2, facecolor='#00008B', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size2} matched via GNPS suspect spectral library')
circle3 = plt.Circle((0.5, 0.165), radius3, facecolor='#ADD8E6', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size3} matched via GNPS non-suspect spectral library')
circle4 = plt.Circle((0.5, 0.14), radius4, facecolor='#0096FF', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size4} non-suspect library + annotation')

# Add the circles to the plot
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.add_patch(circle3)
ax.add_patch(circle4)

# Add percentage text inside the circles
ax.text(0.505, 0.32, f'{percent1:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.505, 0.235, f'{percent2:.1f}%', ha='center', va='center', fontsize=10, color='black')
ax.text(0.505, 0.135, f'{percent3:.1f}%', ha='center', va='center', fontsize=10, color='white')

# Set limits to ensure all circles are fully visible
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Set aspect ratio to be equal to ensure circles are circular
ax.set_aspect('equal')

# Remove all borders
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove ticks and labels
ax.set_xticks([])
ax.set_yticks([])

# Add legend for the circles with their respective sizes and place it outside the plot
ax.legend(loc='upper left', bbox_to_anchor=(-0.26, 0), fontsize=12)

# Add title with better font style and size
plt.title("Metabolites Identified from GNPS2", loc='center', fontsize=18)

# Adjust layout to make room for the legend
plt.tight_layout()

# Save figure
plt.savefig('../Figures/metabolomics_Figures/overview/ms2_venn_diagram.png', dpi=600)



### Sankey plot of GNPS metabolites without suspects at superclass, class, and subclass annotations

In [145]:
# Replace 'and' with '&' in specified columns
columns_to_replace = ['superclass', 'class', 'subclass']
for col in columns_to_replace:
    gnps_without_suspects[col] = gnps_without_suspects[col].str.replace(r'\band\b', '&', regex=True)

# Replace 'Carbohydrate and carbohydrate conjugates' with 'Carbohydrates and conjugates' in the subclass column
gnps_without_suspects['subclass'] = gnps_without_suspects['subclass'].str.replace(
    'Carbohydrates & carbohydrate conjugates', 
    'Carbohydrates & conjugates', 
    regex=False
)
gnps_without_suspects

,SpectrumID,#Scan#,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,SharedPeaks,MassDiff,...,molecular_formula,InChIKey,InChIKey-Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi
0,CCMSLIB00005883855,67,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.997678,3961000.0,0,0.839914,6,0.000099,...,C5H11NO2,KZSNJWFQEVHDMF-BYPYZUCNSA-N,KZSNJWFQEVHDMF,Organic acids & derivatives,Carboxylic acids & derivatives,"Amino acids, peptides, & analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883855
1,CCMSLIB00012249715,2239,spectra_filtered.mgf,CMMC-REFRAME-POSITIVE-LIBRARY.mgf,0.995422,9930000.0,0,2.796090,7,0.000504,...,C6H12O6,LKDRXBCSQODPBY-UHFFFAOYSA-N,LKDRXBCSQODPBY,Organic oxygen compounds,Organooxygen compounds,Carbohydrates & conjugates,Saccharides,Monosaccharides,Carbohydrates,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012249715
2,CCMSLIB00012716670,23882,spectra_filtered.mgf,GNPS-N-ACYL-LIPIDS-MASSQL.mgf,0.992183,2779000.0,0,1.793690,8,0.000488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012716670
3,CCMSLIB00005883686,1368,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.992042,12116000.0,0,2.310150,5,0.000305,...,C6H13NO2,AGPKZVBTJJNPAG-WHFBIAKZSA-N,AGPKZVBTJJNPAG,Organic acids & derivatives,Carboxylic acids & derivatives,"Amino acids, peptides, & analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883686
4,CCMSLIB00005884056,1974,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.991977,2339000.0,0,0.000000,5,0.000000,...,C6H11NO3,UZTFMUBKZQVKLK-UHFFFAOYSA-N,UZTFMUBKZQVKLK,Organic acids & derivatives,Carboxylic acids & derivatives,"Amino acids, peptides, & analogues",Fatty Acids and Conjugates,Unsaturated fatty acids,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005884056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,CCMSLIB00004715189,21585,spectra_filtered.mgf,MONA.mgf,0.605988,350200.0,0,2.549490,5,0.000793,...,C18H30O4,GUVJPXABQYFWPD-GSZDNMEJSA-N,GUVJPXABQYFWPD,Lipids & lipid-like molecules,Prenol lipids,Sesquiterpenoids,Diterpenoids,Labdane diterpenoids,Terpenoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00004715189
428,CCMSLIB00003136040,18597,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.602108,335300.0,0,5.583870,6,0.001190,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003136040
429,CCMSLIB00012443965,11960,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.601783,1953600.0,0,2.178520,10,0.000885,...,C19H32O8,JVIDANAJZDBKRL-FJUMDGKUSA-N,JVIDANAJZDBKRL,Lipids & lipid-like molecules,Prenol lipids,Terpene glycosides,Apocarotenoids,Megastigmanes,Terpenoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012443965
430,CCMSLIB00013558434,12746,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.601377,10408000.0,0,78.356900,5,0.019989,...,C16H14O3,RGSUZUQISVAJJF-UHFFFAOYSA-N,RGSUZUQISVAJJF,Phenylpropanoids & polyketides,Neoflavonoids,Dalbergiones,Flavonoids,Open-chained neoflavonoids,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00013558434


In [146]:
gnps_without_suspects['superclass'].value_counts()

superclass
Lipids & lipid-like molecules              112
Organic acids & derivatives                 41
Benzenoids                                  41
Organic oxygen compounds                    40
Organoheterocyclic compounds                33
Phenylpropanoids & polyketides              24
Organic nitrogen compounds                  20
Lignans, neolignans & related compounds      4
Organophosphorus compounds                   2
Nucleosides, nucleotides, & analogues        1
Name: count, dtype: int64

In [147]:
gnps_without_suspects['superclass'].value_counts().sum()

318

In [148]:
# Number of Nans in superclass
num_nan = gnps_without_suspects['superclass'].isna().sum()

print(f"The column 'superclass' has {num_nan} N/A values.")


The column 'superclass' has 114 N/A values.


In [149]:
# View quality of the spectral match from GNPS
gnps_without_suspects['LibraryQualityString'].value_counts()

LibraryQualityString
Gold        233
Bronze      188
Insilico     10
Silver        1
Name: count, dtype: int64

In [150]:
# Number of inSilico within LibraryQualityString column
num_insilico = len(gnps_without_suspects['LibraryQualityString'] == 'Insilico')

print(f"The column 'LibraryQualityString' has {num_insilico} Insilico values.")


The column 'LibraryQualityString' has 432 Insilico values.


In [151]:
# Remove rows with NaN values in the 'class' or 'subclass' columns
gnps_without_suspects.dropna(subset=['class', 'subclass'], inplace=True)

# Count occurrences for each unique label in superclass, class, and subclass
superclass_counts = gnps_without_suspects['superclass'].value_counts()
class_counts = gnps_without_suspects['class'].value_counts()
subclass_counts = gnps_without_suspects['subclass'].value_counts()

# Filter out categories with fewer than 5 occurrences in class and subclass
gnps_without_suspects = gnps_without_suspects[
    gnps_without_suspects['class'].isin(class_counts[class_counts >= 5].index) &
    gnps_without_suspects['subclass'].isin(subclass_counts[subclass_counts >= 5].index)
]

# Recalculate unique labels after collapsing class and subclass
superclass_labels = gnps_without_suspects['superclass'].unique()  # All categories stay in superclass
class_labels = gnps_without_suspects['class'].unique()  # Collapsed classes
subclass_labels = gnps_without_suspects['subclass'].unique()  # Collapsed subclasses

# Sort superclass categories by their counts in descending order
sorted_superclass_counts = superclass_counts.sort_values(ascending=False)

# Get the sorted order of superclass labels
sorted_superclass_labels = sorted_superclass_counts.index.tolist()

# Recalculate y positions based on sorted superclass labels
superclass_y = [i / len(sorted_superclass_labels) for i in range(len(sorted_superclass_labels))]

# Assign x positions for superclass, class, and subclass categories
superclass_x = [0.3] * len(sorted_superclass_labels)  # Superclass nodes at 30% width
class_y = [i / len(class_labels) for i in range(len(class_labels))]
class_x = [0.6] * len(class_labels)  # Class nodes at 60% width
subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]
subclass_x = [0.9] * len(subclass_labels)  # Subclass nodes at 90% width

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [1]  # Include root and unannotated node positions
node_y = [0.5] + superclass_y + class_y + subclass_y + [0.5]  # Include root and unannotated node positions

# Update the label order to match the sorted superclass labels
sorted_all_labels = ['All classified annotations\n(n=250)'] + sorted_superclass_labels + list(class_labels) + list(subclass_labels)


# Create a dictionary to map labels to indices
label_dict = {label: idx for idx, label in enumerate(sorted_all_labels)}

# Generate source, target, and value lists for the Sankey plot
sources = []
targets = []
values = []

# 'GNPS2 identified metabolites (without suspect library)' -> Superclass
for idx, row in gnps_without_suspects.iterrows():
    source_idx = label_dict['All classified annotations\n(n=250)']
    superclass_idx = label_dict[row['superclass']]
    sources.append(source_idx)
    targets.append(superclass_idx)
    values.append(1)  # Assuming one metabolite per row, otherwise adjust the value

# Superclass -> Class
for idx, row in gnps_without_suspects.iterrows():
    superclass_idx = label_dict[row['superclass']]
    class_idx = label_dict[row['class']]
    sources.append(superclass_idx)
    targets.append(class_idx)
    values.append(1)  # Assuming one metabolite per row

# Class -> Subclass
for idx, row in gnps_without_suspects.iterrows():
    class_idx = label_dict[row['class']]
    subclass_idx = label_dict[row['subclass']]
    sources.append(class_idx)
    targets.append(subclass_idx)
    values.append(1)  # Again, assuming one metabolite per row

# Generate y positions for nodes based on their category
superclass_y = [i / len(superclass_labels) for i in range(len(superclass_labels))]
class_y = [i / len(class_labels) for i in range(len(class_labels))]
subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]

# Generate x positions for each category of nodes
superclass_x = [0.3] * len(superclass_labels)  # Superclass nodes at 30% width
class_x = [0.6] * len(class_labels)            # Class nodes at 60% width
subclass_x = [0.9] * len(subclass_labels)      # Subclass nodes at 90% width

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [1]  # Add positions for the root and unannotated node
node_y = [0.5] + superclass_y + class_y + subclass_y + [0.5]  # Add positions for the root and unannotated node

# Get the 'tab10' colormap
tab10 = colormaps['tab10']

# Skip the first color and scale the remaining colors based on the number of superclass labels
# superclass_palette = [tab10(i / (len(superclass_labels) + 1)) for i in range(1, len(superclass_labels) + 1)]
superclass_palette = [tab10(i / 9) for i in range(1, len(sorted_superclass_labels) + 1)]  # Scale indices appropriately

# Convert to hex and assign to labels
superclass_colors = {label: mcolors.to_hex(superclass_palette[i]) for i, label in enumerate(sorted_superclass_labels)}

# Generate colors for classes and subclasses based on superclass colors
node_colors = []
for label in sorted_all_labels:
    if label in superclass_labels:
        node_colors.append(superclass_colors[label])  # Use superclass color
    elif label in class_labels:
        # Derive lighter shades from superclass color
        superclass = gnps_without_suspects[gnps_without_suspects['class'] == label].iloc[0]['superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[superclass], alpha=0.8)))
    elif label in subclass_labels:
        # Derive even lighter shades from class color
        related_class = gnps_without_suspects[gnps_without_suspects['subclass'] == label].iloc[0]['class']
        related_superclass = gnps_without_suspects[gnps_without_suspects['subclass'] == label].iloc[0]['superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[related_superclass], alpha=0.6)))
    else:
        node_colors.append('#ededed')  # Default color for "Unannotated" and "All classified annotations"

# Create the Sankey plot with spaced-out nodes and consistent colors
fig = go.Figure(go.Sankey(
    node=dict(
        pad=200,
        thickness=10,
        line=dict(color='black', width=0.5),
        label=sorted_all_labels,
        x=node_x,
        y=node_y,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color='rgba(58, 158, 224, 0.3)'  #3a9ee0 blue color
    )
))

fig.update_layout(
    title=dict(
        text="Annotated Categories of GNPS2 Metabolites",
        x=0.5,  # Center the title
        xanchor='center',  # Align the title by its center
        font=dict(
            size=20,  # Adjust title font size
            color='black'  # Set the title text color (your desired color)
        )
    ),
    font=dict(size=13),  # General font size for other elements
    width=1000,
    height=500,
    margin=dict(l=10, r=225, t=50, b=10)
)

# Save the figure as PNG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_plot.png", format="png", width=1000, height=500, scale=2)

# Save the figure as SVG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_plot.svg", format="svg", width=1000, height=500)

fig.show()

## Upset plot of MASST categories

In [ ]:
# Import MASST results file
file_path = "../Data/metabolomics/Run3_10252024/masst_screening_results_for-upset.csv" # This file is masst_screening_results.csv but with the Humans and Human Skin categories converted from TRUE/False to 1/0

masst_categories = pd.read_csv(file_path)

# Add a new column named Microbes to your masst_categories
masst_categories['Microbes'] = masst_categories['Category'].apply(lambda x: 1 if x == "microbes" else 0)

# Add a new column named Microbes to your masst_categories
masst_categories['Personal Care Product'] = masst_categories['Category'].apply(lambda x: 1 if x == "personal care product" else 0)

masst_categories.to_csv("../Data/metabolomics/Run3_10252024/masst_screening_results_for-upset_final.csv")


In [159]:
# File path for input and output
file_path = "../Data/metabolomics/Run3_10252024/masst_screening_results_for-upset_final.csv"
output_file = "../Figures/metabolomics_Figures/overview/upset_plot_microbes_human-skin_pcp.png"

# Load the dataset
masst_categories = pd.read_csv(file_path)

# Columns of interest
columns_of_interest = ['Human Skin', 'Microbes', 'Personal Care Product']
data_subset = masst_categories[columns_of_interest].astype(bool)

# Count rows where all three columns have a value of 1
count_all_three = data_subset[(data_subset['Human Skin']) & 
                              (data_subset['Microbes']) & 
                              (data_subset['Personal Care Product'])].shape[0]

# Print the result
print(f"Number of rows where all three columns are 1: {count_all_three}")

Number of rows where all three columns are 1: 0


In [161]:
# Prepare the data for the upset plot
columns_of_interest = ['Human Skin', 'Microbes', 'Personal Care Product']
data_subset = masst_categories[columns_of_interest]

# Convert columns to boolean format
data_subset = data_subset.astype(bool)

# Generate a multi-index from the boolean DataFrame
memberships = []
for idx, row in data_subset.iterrows():
    memberships.append(
        [col for col, value in row.items() if value]
    )

# Create the UpSet data
upset_data = from_memberships(memberships)

# Create the UpSet plot
upset = UpSet(upset_data, subset_size='count', show_counts=True, sort_by='cardinality')
upset.plot()

# Save the plot to the specified file path
plt.savefig(output_file, dpi=600)


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/upsetplot/data.py:303: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/upsetplot/plotting.py:795: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.